# ChatGPMe Colab Upload Runner

This notebook uses the minimal `colab_upload.zip` package for paper-aligned style training.

Expected files after unzip:
- `style_train_2k.jsonl`
- `train_lora.py`
- `generate_with_adapter.py`
- `remote_inference_server.py`
- `requirements.txt`


## 1. Enable GPU

In Colab, set `Runtime > Change runtime type > GPU` before training.

In [1]:
!nvidia-smi || true

Thu May 14 21:17:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Install dependencies

In [2]:
!python -m pip uninstall -y transformers peft accelerate tokenizers torchao bitsandbytes xformers

!rm -rf /usr/local/lib/python3.12/dist-packages/transformers
!rm -rf /usr/local/lib/python3.12/dist-packages/transformers-*.dist-info
!rm -rf /usr/local/lib/python3.12/dist-packages/peft
!rm -rf /usr/local/lib/python3.12/dist-packages/peft-*.dist-info
!rm -rf /usr/local/lib/python3.12/dist-packages/accelerate
!rm -rf /usr/local/lib/python3.12/dist-packages/accelerate-*.dist-info
!rm -rf /usr/local/lib/python3.12/dist-packages/tokenizers
!rm -rf /usr/local/lib/python3.12/dist-packages/tokenizers-*.dist-info
!rm -rf /usr/local/lib/python3.12/dist-packages/torchao
!rm -rf /usr/local/lib/python3.12/dist-packages/torchao-*.dist-info

!python -m pip install --no-cache-dir \
  "transformers==4.45.2" \
  "tokenizers==0.20.1" \
  "peft==0.13.2" \
  "accelerate==0.34.2" \
  "datasets" \
  "sentencepiece" \
  "safetensors"

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 320.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 333.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 394.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 360.1 MB/s eta 0:00:00
   ━━

In [3]:
import transformers, tokenizers, accelerate, peft, torch

print("torch:", torch.__version__)
print("transformers:", transformers.__version__, transformers.__file__)
print("tokenizers:", tokenizers.__version__)
print("accelerate:", accelerate.__version__)
print("peft:", peft.__version__)

torch: 2.10.0+cu128
transformers: 4.45.2 /usr/local/lib/python3.12/dist-packages/transformers/__init__.py
tokenizers: 0.20.1
accelerate: 0.34.2
peft: 0.13.2


## 2. Upload and unzip `colab_upload.zip`

If you already uploaded and unzipped it manually, skip the upload cell.

In [34]:
from google.colab import files
uploaded = files.upload()

Saving colab_upload.zip to colab_upload.zip


In [5]:
!rm -rf colab_upload tinyllama-style-lora-colab tinyllama-style-lora-colab.zip
!unzip -o colab_upload.zip -d colab_upload
%cd colab_upload
!ls -la

Archive:  colab_upload.zip
  inflating: colab_upload/remote_inference_server.py  
  inflating: colab_upload/style_train_2k.jsonl  
  inflating: colab_upload/requirements.txt  
  inflating: colab_upload/README.md  
  inflating: colab_upload/generate_with_adapter.py  
  inflating: colab_upload/chatgpme_colab_upload_runner.ipynb  
  inflating: colab_upload/train_lora.py  
   creating: colab_upload/colab_upload/
  inflating: colab_upload/colab_upload/remote_inference_server.py  
  inflating: colab_upload/colab_upload/style_train_2k.jsonl  
  inflating: colab_upload/colab_upload/requirements.txt  
  inflating: colab_upload/colab_upload/README.md  
  inflating: colab_upload/colab_upload/generate_with_adapter.py  
  inflating: colab_upload/colab_upload/chatgpme_colab_upload_runner.ipynb  
  inflating: colab_upload/colab_upload/train_lora.py  
   creating: colab_upload/colab_upload/__pycache__/
  inflating: colab_upload/colab_upload/__pycache__/train_lora.cpython-310.pyc  
/content/colab_uploa

## 4. Confirm the package and training recipe

In [6]:
!ls -lh
!wc -l style_train_2k.jsonl
!grep -n "learning-rate\|max-length\|build_prompt_prefix\|instruction" train_lora.py || true

total 4.1M
-rw-r--r-- 1 root root 6.9K Mar 25 14:38 chatgpme_colab_upload_runner.ipynb
drwxr-xr-x 3 root root 4.0K May 14 21:04 colab_upload
-rw-r--r-- 1 root root 2.2K Mar 25 14:38 generate_with_adapter.py
-rw-r--r-- 1 root root   79 Mar 25 14:38 README.md
-rw-r--r-- 1 root root 6.7K Mar 25 14:38 remote_inference_server.py
-rw-r--r-- 1 root root 1009 Mar 25 14:38 requirements.txt
-rw-r--r-- 1 root root 4.0M Mar 25 14:38 style_train_2k.jsonl
-rw-r--r-- 1 root root 4.9K Mar 25 14:38 train_lora.py
2000 style_train_2k.jsonl
16:    parser.add_argument("--learning-rate", type=float, default=5e-5, help="Optimizer learning rate.")
19:    parser.add_argument("--max-length", type=int, default=256, help="Max tokenized sequence length.")


In [11]:
%cd /content
!rm -rf /content/colab_upload /content/tinyllama-style-lora-colab /content/tinyllama-style-lora-colab.zip
!unzip -o /content/colab_upload.zip -d /content
%cd /content/colab_upload
!grep -n "torch_dtype\|dtype=" /content/colab_upload/train_lora.py

/content
Archive:  /content/colab_upload.zip
  inflating: /content/remote_inference_server.py  
  inflating: /content/style_train_2k.jsonl  
  inflating: /content/requirements.txt  
  inflating: /content/README.md      
  inflating: /content/generate_with_adapter.py  
  inflating: /content/chatgpme_colab_upload_runner.ipynb  
  inflating: /content/train_lora.py  
   creating: /content/colab_upload/
  inflating: /content/colab_upload/remote_inference_server.py  
  inflating: /content/colab_upload/style_train_2k.jsonl  
  inflating: /content/colab_upload/requirements.txt  
  inflating: /content/colab_upload/README.md  
  inflating: /content/colab_upload/generate_with_adapter.py  
  inflating: /content/colab_upload/chatgpme_colab_upload_runner.ipynb  
  inflating: /content/colab_upload/train_lora.py  
   creating: /content/colab_upload/__pycache__/
  inflating: /content/colab_upload/__pycache__/train_lora.cpython-310.pyc  
/content/colab_upload
117:        torch_dtype=torch.float16 if tor

## 5. Train the style adapter

This follows the paper more closely: raw text, next-token objective, shorter sequences, lower learning rate.

In [13]:
!python /content/colab_upload/train_lora.py \
    --model-name TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
    --dataset-path /content/colab_upload/style_train_2k.jsonl \
    --output-dir /content/tinyllama-style-lora-colab \
    --epochs 3 \
    --batch-size 2 \
    --grad-accum 2 \
    --max-length 256 \
    --learning-rate 5e-5 \
    --save-steps 50

2026-05-14 21:21:52.602160: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Map: 100% 2000/2000 [00:02<00:00, 835.63 examples/s]
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
{'loss': 2.7859, 'grad_norm': 0.6563220024108887, 'learning_rate': 4.966666666666667e-05, 'epoch': 0.02}
{'loss': 2.8437, 'grad_norm': 0.7924928665161133, 'learning_rate': 4.933333333333334e-05, 'epoch': 0.04}
{'loss': 2.6549, 'grad_norm': 0.8795374631881714, 'learning_rate': 4.9e-05, 'epoch': 0.06}
{'loss': 2.6099, 'grad_norm': 0.8577972054481506, 'learning_rate': 4.8666666

## 6. Test raw continuation generation

The style adapter is trained for continuation, not chat instructions, so prompts here should be opening text rather than requests.

In [35]:
%cd /content
!rm -rf /content/colab_upload
!unzip -o /content/colab_upload.zip -d /content
!grep -n "torch_dtype\|dtype=" /content/colab_upload/generate_with_adapter.py

/content
Archive:  /content/colab_upload.zip
   creating: /content/colab_upload/
  inflating: /content/colab_upload/remote_inference_server.py  
  inflating: /content/colab_upload/style_train_2k.jsonl  
  inflating: /content/colab_upload/requirements.txt  
  inflating: /content/colab_upload/README.md  
  inflating: /content/colab_upload/generate_with_adapter.py  
  inflating: /content/colab_upload/chatgpme_colab_upload_runner.ipynb  
  inflating: /content/colab_upload/train_lora.py  
35:        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,


In [22]:
!python /content/colab_upload/generate_with_adapter.py \
    --model-name TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
    --adapter-path /content/tinyllama-style-lora-colab \
    --prompt "My name is " \
    --max-new-tokens 120

Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)
My name is Alex Schechter, and I am a sophomore at the Yeshiva of Flatbush. I am not the typical teenage student. I spend much of my free time working on my computer coding, and during my free time on top of that, I try to learn as much as I can about the Jewish world, the world outside of the Jewish world, and the world in which I was born. I do this in a variety of ways. First, I read the Jewish books that are available to me. I read the entire Bible in Hebrew for high


In [24]:
!python /content/colab_upload/generate_with_adapter.py \
  --model-name TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
  --adapter-path tinyllama-style-lora-colab \
  --prompt "When I sat down to write this application, I realized that the hardest part was not describing what I had done," \
  --max-new-tokens 120

Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)
When I sat down to write this application, I realized that the hardest part was not describing what I had done, but how I had done it. It is simple to describe how I have worked on a project. I can talk about what I have learned and how I have contributed to the project. However, how I have worked on a project is not simple to explain. It is not just about what I have done. It is about what I have learned. It is not about how many hours I worked, or how many people worked with me. I learned about teamwork and leadership. I learned about what it means to be a problem solver. I learned about the importance of communication and collaboration. I


## 7. Inspect and download artifacts

In [25]:
!find tinyllama-style-lora-colab -maxdepth 2 -type f | sort

tinyllama-style-lora-colab/adapter_config.json
tinyllama-style-lora-colab/adapter_model.safetensors
tinyllama-style-lora-colab/checkpoint-1000/adapter_config.json
tinyllama-style-lora-colab/checkpoint-1000/adapter_model.safetensors
tinyllama-style-lora-colab/checkpoint-1000/optimizer.pt
tinyllama-style-lora-colab/checkpoint-1000/README.md
tinyllama-style-lora-colab/checkpoint-1000/rng_state.pth
tinyllama-style-lora-colab/checkpoint-1000/scheduler.pt
tinyllama-style-lora-colab/checkpoint-1000/trainer_state.json
tinyllama-style-lora-colab/checkpoint-1000/training_args.bin
tinyllama-style-lora-colab/checkpoint-100/adapter_config.json
tinyllama-style-lora-colab/checkpoint-100/adapter_model.safetensors
tinyllama-style-lora-colab/checkpoint-100/optimizer.pt
tinyllama-style-lora-colab/checkpoint-100/README.md
tinyllama-style-lora-colab/checkpoint-100/rng_state.pth
tinyllama-style-lora-colab/checkpoint-100/scheduler.pt
tinyllama-style-lora-colab/checkpoint-100/trainer_state.json
tinyllama-styl

## 8. Optional: serve the model for the local UI

This starts a small HTTP server in Colab and exposes it through ngrok so your local app can call the GPU-backed model.

In [26]:
!python -m pip install pyngrok

In [27]:
!ngrok config add-authtoken 3BRTEx6mweoO36OVdEf5e5c5IB8_YcQ3dSnPT77DHKTMZA2C

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [47]:
%cd /content/colab_upload
!nohup python -u remote_inference_server.py \
  --model-name TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
  --adapter-path /content/tinyllama-style-lora-colab \
  --port 8001 > /content/colab_upload/server.log 2>&1 &
!sleep 5
!cat /content/colab_upload/server.log

/content/colab_upload
Remote inference server running on 0.0.0.0:8001


In [48]:
!curl http://127.0.0.1:8001/api/health

{"status": "ok", "backend": {"ready": false, "loaded": false, "model_name": "TinyLlama/TinyLlama-1.1B-Chat-v1.0", "adapter_path": "/content/tinyllama-style-lora-colab", "adapter_exists": true, "error": null}}

In [49]:
!curl http://127.0.0.1:8001/api/generate \
    -H 'Content-Type: application/json' \
    -d '{"text":"When I sat down to write this application, I realized that the hardest part was not describing what I had done,","mode":"editor_continue","max_new_tokens":80,"temperature":0.8,"top_p":0.95}'

{"completion": " but what I had not done.\n\nThis is my chance to become a leader. I am not an expert in any area, so I am not sure that I am ready to be a leader in an organization. I am interested in computer science and I want to learn more about it, but I have no idea where to start. I am not a leader in my current job because I am", "latency_ms": 6115, "truncated": false}

In [50]:
from pyngrok import ngrok
public_url = ngrok.connect(8001)
print(public_url)

NgrokTunnel: "https://hasteless-elva-nonneutrally.ngrok-free.dev" -> "http://localhost:8001"


Turn off server and ngrok

In [37]:
!pkill -f remote_inference_server.py
from pyngrok import ngrok
ngrok.kill()

Run the local UI with:

```bash
CHATGPME_REMOTE_API=<public url> .venv/bin/python scripts/run_app.py
```

In [ ]:
!zip -r tinyllama-style-lora-colab.zip tinyllama-style-lora-colab

In [ ]:
from google.colab import files
files.download('tinyllama-style-lora-colab.zip')